In [38]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor

In [40]:
# Dosya yolu
df = pd.read_csv("all_data_ratio.csv")

In [42]:
# Feature map
feature_map = {
    "uzman": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "diger"],
    "pratisyen": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "ebe", "diger"],
    "asistan": ["yıl", "nufus", "hastane", "yatak", "uzman", "pratisyen", "hemsire", "ebe", "diger"],
    "ebe": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "hemsire", "pratisyen", "diger"],
    "diger": ["yıl", "nufus", "hastane", "yatak", "asistan", "pratisyen", "hemsire", "ebe", "uzman"],
    "hemsire": ["yıl", "nufus", "hastane", "yatak", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "yatak": ["yıl", "nufus", "hastane", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"],
    "hastane": ["yıl", "nufus", "yatak", "hemsire", "asistan", "uzman", "pratisyen", "ebe", "diger"]
}

In [44]:
tahminler = []

for il in df["il_adı"].unique():
    df_il = df[df["il_adı"] == il]
    if df_il[df_il["yıl"] <= 2021].shape[0] < 5:
        continue

    for hedef, features in feature_map.items():
        if hedef not in df_il.columns or any(f not in df_il.columns for f in features):
            continue

        df_target = df_il[features + [hedef]].dropna()
        if df_target.empty or df_target[df_target["yıl"] <= 2021].shape[0] == 0:
            continue

        train = df_target[df_target["yıl"] <= 2021]
        test = df_target[df_target["yıl"] == 2023]
        if test.empty:
            continue

        X_train = train[features]
        y_train = train[hedef]
        X_test = test[features]
        y_test = test[hedef]

        # 6 Modelin Eğitimi
        lr = LinearRegression().fit(X_train, y_train)
        rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(X_train, y_train)
        xgb = XGBRegressor(objective='reg:squarederror', random_state=42).fit(X_train, y_train)
        svr = SVR().fit(X_train, y_train)
        knn = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)

        # Tahminler
        tahminler.append({
            "il_adı": il,
            "hedef": hedef,
            "gercek_2023": y_test.values[0],
            "tahmin_2023_lr": lr.predict(X_test)[0],
            "tahmin_2023_rf": rf.predict(X_test)[0],
            "tahmin_2023_xgb": xgb.predict(X_test)[0],
            "tahmin_2023_svr": svr.predict(X_test)[0],
            "tahmin_2023_knn": knn.predict(X_test)[0],
        })

In [48]:
df_tahmin = pd.DataFrame(tahminler)

In [50]:
# Doğruluk hesaplama
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    df_tahmin[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df_tahmin["gercek_2023"] - df_tahmin[f"tahmin_2023_{model}"]) / df_tahmin["gercek_2023"] * 100
    )
    df_tahmin[f"{model}_yuzde_dog"] = df_tahmin[f"{model}_yuzde_dog"].clip(0, 100)

# Ortalama doğrulukları yazdır
print("\nŞehir Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ort = round(df_tahmin[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ort}%")

# Doğruluk dosyasını kaydet
df_tahmin.to_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv", index=False)


Şehir Doğruluk Ortalamaları (2023):
 - LR: 67.81%
 - RF: 78.76%
 - XGB: 79.81%
 - SVR: 69.81%
 - KNN: 78.4%


In [52]:
# Türkiye'nin 7 coğrafi bölgesi
bolge_haritasi = {
    'Marmara': ["İstanbul", "Edirne", "Kırklareli", "Tekirdağ", "Kocaeli", "Sakarya", "Yalova", "Bilecik", "Balıkesir", "Çanakkale", "Bursa"],
    'Ege': ["İzmir", "Aydın", "Muğla", "Manisa", "Afyonkarahisar", "Uşak", "Kütahya", "Denizli"],
    'Akdeniz': ["Antalya", "Mersin", "Adana", "Hatay", "Kahramanmaraş", "Osmaniye", "Isparta", "Burdur"],
    'İç Anadolu': ["Ankara", "Konya", "Kayseri", "Eskişehir", "Nevşehir", "Aksaray", "Kırıkkale", "Niğde", "Karaman", "Sivas", "Yozgat", "Kırşehir", "Çankırı"],
    'Karadeniz': ["Samsun", "Trabzon", "Ordu", "Giresun", "Rize", "Artvin", "Tokat", "Amasya", "Bartın", "Zonguldak", "Karabük", "Sinop", "Çorum", "Bayburt", "Gümüşhane", "Kastamonu", "Bolu", "Düzce"],
    'Doğu Anadolu': ["Erzurum", "Van", "Malatya", "Elazığ", "Kars", "Ağrı", "Bitlis", "Iğdır", "Tunceli", "Ardahan", "Erzincan", "Bingöl", "Muş", "Hakkari"],
    'Güneydoğu Anadolu': ["Diyarbakır", "Gaziantep", "Şanlıurfa", "Mardin", "Batman", "Siirt", "Şırnak", "Adıyaman", "Kilis"]
}

def bolge_bul(il):
    for bolge, iller in bolge_haritasi.items():
        if il in iller:
            return bolge
    return "Bilinmiyor"

# Bölge ekle
df_tahmin["bolge"] = df_tahmin["il_adı"].apply(bolge_bul)

# Bölgelere göre grupla ve toplamları al
bolgesel = df_tahmin.groupby(["bolge", "hedef"])[[
    "gercek_2023",
    "tahmin_2023_lr",
    "tahmin_2023_rf",
    "tahmin_2023_xgb",
    "tahmin_2023_svr",
    "tahmin_2023_knn"
]].sum().reset_index()

# CSV olarak kaydet
bolgesel.to_csv("bolge_bazli_2023_tahmin5model.csv", index=False)

In [60]:
# 1. Dosyayı oku
df = pd.read_csv("sehir_bazli_2023_tahmin_dogruluk_5model.csv")  # Gerekirse yolunu düzenle

# 2. Doğruluk hesaplama fonksiyonu
def hesapla_dog(model_etiketi):
    df[f"{model_etiketi}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model_etiketi}"]) / df["gercek_2023"] * 100
    )
    df[f"{model_etiketi}_yuzde_dog"] = df[f"{model_etiketi}_yuzde_dog"].clip(0, 100)

# 3. Altı model için hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    hesapla_dog(model)

# 4. Ortalamaları yazdır
print("\nŞehir Bazlı Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 5. Güncel dosya olarak kaydet
df.to_csv("sehir_tahmin_yuzdeler.csv", index=False)


Şehir Bazlı Doğruluk Ortalamaları (2023):
 - LR: 67.81%
 - RF: 78.76%
 - XGB: 79.81%
 - SVR: 69.81%
 - KNN: 78.4%


In [62]:
# 1. Dosyayı oku
df = pd.read_csv("bolge_bazli_2023_tahmin6model.csv")  # Gerekirse dosya adını değiştir

# 2. Her model için yüzdelik doğruluk hesapla
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    df[f"{model}_yuzde_dog"] = 100 - (
        np.abs(df["gercek_2023"] - df[f"tahmin_2023_{model}"]) / df["gercek_2023"] * 100
    )
    df[f"{model}_yuzde_dog"] = df[f"{model}_yuzde_dog"].clip(0, 100)

# 3. Ortalamaları hesapla ve yazdır
print(f"\nBölge Doğruluk Ortalamaları (2023):")
for model in ["lr", "rf", "xgb", "svr", "knn"]:
    ortalama = round(df[f"{model}_yuzde_dog"].mean(), 2)
    print(f" - {model.upper()}: {ortalama}%")

# 4. Dosyaya kaydet
df.to_csv("bolge_tahmin_yuzdeler.csv", index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'bolge_bazli_2023_tahmin6model.csv'